In [1]:
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rich import print, pretty, inspect
from rich.console import Console

In [2]:
console = Console()
console.print("hello", style = "bold white")

hello

In [3]:
wrappers_root = Path(".")

target_file = "casper_wb_fft_config.m"
target_fpath = wrappers_root / target_file

### Extract File dependencies 
1. Regex patterns
    1. Find import lines: ```this_block.addFileToLibrary(...);``` https://regex101.com/r/gilGkt/1
    2. Match and extract file paths: 

In [4]:
dep_pat_import_lines = "this_block.addFileToLibrary\((.*),.*'(.*)'\);"
dep_pat_fpaths = "filepath '(.*)']"
relative_path_pat = "[/\.]*(.*)"
scilab_casper_prefix = "casper_dspdevel"

with open(target_fpath, "r") as f:
    d = f.read()

matches = re.findall(dep_pat_import_lines, d)
matches[:5]

[('vhdlfile', 'xil_defaultlib'),
 ("[filepath '/../../common_pkg/fixed_float_types_c.vhd']", 'common_pkg_lib'),
 ("[filepath '/../../common_pkg/fixed_pkg_c.vhd']", 'common_pkg_lib'),
 ("[filepath '/../../common_pkg/common_pkg.vhd']", 'common_pkg_lib'),
 ("[filepath '/../../common_components/common_pipeline.vhd']",
  'common_components_lib')]

In [5]:
dep_df = pd.DataFrame(matches).rename(columns={0: "simulink_fpath", 1: "lib"})
dep_df['simulink_fpath'] = dep_df['simulink_fpath'].str.extract(dep_pat_fpaths)
dep_df = dep_df.dropna().reset_index(drop=True)
dep_df['scilab_fpath'] = scilab_casper_prefix + "/" + (dep_df['simulink_fpath'].str.extract(relative_path_pat))
dep_df

,simulink_fpath,lib,scilab_fpath
0,/../../common_pkg/fixed_float_types_c.vhd,common_pkg_lib,casper_dspdevel/common_pkg/fixed_float_types_c...
1,/../../common_pkg/fixed_pkg_c.vhd,common_pkg_lib,casper_dspdevel/common_pkg/fixed_pkg_c.vhd
2,/../../common_pkg/common_pkg.vhd,common_pkg_lib,casper_dspdevel/common_pkg/common_pkg.vhd
3,/../../common_components/common_pipeline.vhd,common_components_lib,casper_dspdevel/common_components/common_pipel...
4,/../../casper_adder/common_add_sub.vhd,casper_adder_lib,casper_dspdevel/casper_adder/common_add_sub.vhd
5,/../../common_components/common_async.vhd,common_components_lib,casper_dspdevel/common_components/common_async...
6,/../../common_components/common_areset.vhd,common_components_lib,casper_dspdevel/common_components/common_arese...
7,/../../common_components/common_bit_delay.vhd,common_components_lib,casper_dspdevel/common_components/common_bit_d...
8,/../../common_components/common_pipeline_sl.vhd,common_components_lib,casper_dspdevel/common_components/common_pipel...
9,/../../casper_multiplier/tech_mult_component.vhd,casper_multiplier_lib,casper_dspdevel/casper_multiplier/tech_mult_co...


In [7]:
# In Jupyter notebook
from IPython.display import Image
import ast
from graphviz import Digraph

def extract_ast(code):
    """Extract abstract syntax tree from Python code"""
    tree = ast.parse(code)
    return tree

def visualize_ast(tree):
    """Create Graphviz visualization of AST"""
    dot = Digraph(comment='AST', node_attr={'shape': 'box', 'style': 'filled', 'fillcolor': '#f0f0f0'})
    
    def add_nodes_edges(node, parent=None):
        node_name = str(id(node))
        label = type(node).__name__
        
        # Add special handling for common node types
        if isinstance(node, ast.ClassDef):
            dot.node(node_name, f"Class: {node.name}", fillcolor='#e0f0e0')
        elif isinstance(node, ast.FunctionDef):
            dot.node(node_name, f"Function: {node.name}", fillcolor='#e0e0f0')
        elif isinstance(node, ast.Name):
            dot.node(node_name, f"Name: {node.id}", fillcolor='#f0e0f0')
        else:
            dot.node(node_name, label)
            
        if parent:
            dot.edge(parent, node_name)
            
        for child in ast.iter_child_nodes(node):
            add_nodes_edges(child, node_name)
            
    add_nodes_edges(tree)
    return dot

# Usage example
with open('simulink/vivadoprojconfigparse.py', 'r') as f:
    code = f.read()

# 1. Extract AST
ast_tree = extract_ast(code)
print(ast.dump(ast_tree, indent=4))  # Text representation

# 2. Visualize AST
dot = visualize_ast(ast_tree)
dot.render('ast_graph', format='png', cleanup=True)  # Save as PNG
dot  # Display in notebook if using Jupyter

Image(dot.render(format='png'))


ModuleNotFoundError: No module named 'graphviz'